In [1]:
"""
=============================================================================
Breast Cancer Detection — Mammogram Module (Local Machine)
Model   : MobileNetV3-Large (Transfer Learning)
Dataset : CBIS-DDSM (Kaggle: awsaf49/cbis-ddsm-breast-cancer-image-dataset)
Output  : INT8 TFLite quantized model
Author  : Mark Sthembiso Mando | Mulungushi University MSc Data Science
=============================================================================
 
SETUP — run once in WSL2 terminal:
    pip install tensorflow[and-cuda] opencv-python scikit-learn matplotlib pandas tqdm
 
ONE-TIME DATASET COPY (WSL2 only — moves data to fast native Linux filesystem):
    cp -r /mnt/c/Users/mmando/Downloads/CBIS-DDSM ~/CBIS-DDSM
 
HOW TO RUN:
    source ~/tf-env/bin/activate
    python train_mammogram_local.py
 
EXPECTED DATASET STRUCTURE after unzipping Kaggle download:
    CBIS-DDSM/
    ├── csv/
    │   ├── dicom_info.csv
    │   ├── mass_case_description_train_set.csv
    │   ├── mass_case_description_test_set.csv
    │   ├── calc_case_description_train_set.csv
    │   └── calc_case_description_test_set.csv
    └── jpeg/
        └── ... (renamed image folders)
=============================================================================
"""
import os
import re
import glob
import random
import hashlib
import warnings
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe for scripts
import matplotlib.pyplot as plt
from tqdm import tqdm
 
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, mixed_precision
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
 
warnings.filterwarnings("ignore")

I0000 00:00:1776140908.541068   20547 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776140909.229447   20547 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776140911.205897   20547 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# =============================================================================
# 1. CONFIGURATION  ← Only edit this section
# =============================================================================
 
# ── Cross-platform path configuration ────────────────────────────────────────
# DATASET_ROOT : Where your images live.
#   Windows : C:\Users\mmando\Downloads\CBIS-DDSM
#   WSL2    : ~/CBIS-DDSM  (native Linux fs — much faster I/O than /mnt/c/)
#             Copy once: cp -r /mnt/c/Users/mmando/Downloads/CBIS-DDSM ~/CBIS-DDSM
#
# OUTPUT_DIR : Where models/plots/TFLite are saved.
#   WSL2    : ~/mammogram_output  (native Linux fs — fast saves)
#             Copy to Windows when needed:
#             cp -r ~/mammogram_output /mnt/c/Users/mmando/Documents/School/ML/Output
 
if os.name == "nt":                              # Native Windows
    DATASET_ROOT = r"C:\Users\mmando\Downloads\CBIS-DDSM"
    OUTPUT_DIR   = r"C:\Users\mmando\Documents\School\ML\Output"
else:                                            # WSL2 / Linux
    DATASET_ROOT = os.path.expanduser("~/CBIS-DDSM")
    OUTPUT_DIR   = os.path.expanduser("~/mammogram_output")
 
# ── Training hyperparameters (proposal Section 4.4.2) ────────────────────────
IMG_SIZE        = 224       # MobileNetV3 standard input
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 100       # Phase 1: frozen convolutional base
EPOCHS_FINETUNE = 30        # Phase 2: unfreeze last 30 layers
LEARNING_RATE   = 0.0001    # Adam initial LR
DROPOUT_RATE    = 0.3
L2_DECAY        = 0.0001
PATIENCE_STOP   = 15        # EarlyStopping patience
PATIENCE_LR     = 10        # ReduceLROnPlateau patience
SEED            = 42
 
# ── Data split (proposal Section 4.3) ────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
 
# ── TFLite (proposal Section 4.5.1) ──────────────────────────────────────────
TFLITE_CALIBRATION_SAMPLES = 1000
 
# =============================================================================
# Derived paths & setup — do not edit below this line
# =============================================================================
 
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
 
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_DIR          = os.path.join(OUTPUT_DIR, "cache")
CHECKPOINT_PATH_P1 = os.path.join(OUTPUT_DIR, "best_mobilenetv3_phase1.keras")
CHECKPOINT_PATH_P2 = os.path.join(OUTPUT_DIR, "best_mobilenetv3_phase2.keras")
CHECKPOINT_PATH    = CHECKPOINT_PATH_P2   # alias used downstream
TFLITE_PATH        = os.path.join(OUTPUT_DIR, "mammogram_mobilenetv3_int8.tflite")
PLOT_DIR           = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
 
print(f"\n{'='*62}")
print("  CBIS-DDSM  |  MobileNetV3-Large  |  INT8 TFLite")
print(f"{'='*62}\n")
print(f"  Platform   : {'Windows' if os.name == 'nt' else 'WSL2/Linux'}")
print(f"  TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"  GPUs found : {len(gpus)}")
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f"  Using      : {gpus[0].name}")
    print(f"  Precision  : mixed_float16 (Tensor Core acceleration)")
else:
    print("  WARNING    : No GPU detected — training will be slow.")
    print("               Run in WSL2 for GPU support.")
print(f"  Dataset    : {DATASET_ROOT}")
print(f"  Output     : {OUTPUT_DIR}")
print()


  CBIS-DDSM  |  MobileNetV3-Large  |  INT8 TFLite

  Platform   : WSL2/Linux
  TensorFlow : 2.21.0
  GPUs found : 1
  Using      : /physical_device:GPU:0
  Precision  : mixed_float16 (Tensor Core acceleration)
  Dataset    : /home/mmando/CBIS-DDSM
  Output     : /home/mmando/mammogram_output



In [3]:
# =============================================================================
# 2. DATA LOADING
#    Uses dicom_info.csv as the bridge between descriptor CSV DICOM paths
#    and the actual renamed JPEGs in the Kaggle zip (e.g. 2-096.jpg)
# =============================================================================
 
def load_dataset() -> pd.DataFrame:
    """
    Loads CBIS-DDSM descriptor CSVs and resolves each row to a real image
    path via dicom_info.csv (SeriesInstanceUID lookup).
 
    Returns a DataFrame with columns: img_path, pathology, label
    """
 
    # ── Step 1: Load dicom_info.csv ──────────────────────────────────────────
    dicom_info_path = os.path.join(DATASET_ROOT, "csv", "dicom_info.csv")
    if not os.path.exists(dicom_info_path):
        matches = glob.glob(
            os.path.join(DATASET_ROOT, "**", "dicom_info.csv"), recursive=True
        )
        if not matches:
            raise FileNotFoundError(
                f"dicom_info.csv not found inside {DATASET_ROOT}.\n"
                "Ensure the Kaggle zip extracted correctly."
            )
        dicom_info_path = matches[0]
 
    dicom_info = pd.read_csv(dicom_info_path)
    print(f"dicom_info.csv : {len(dicom_info):,} rows")
 
    # ── Step 2: Resolve image_path → full local path ──────────────────────────
    def resolve_image_path(raw: str) -> str:
        raw = re.sub(r"^CBIS-DDSM[/\\]", "", str(raw).strip())
        candidate = os.path.join(DATASET_ROOT, raw)
        return candidate if os.path.exists(candidate) else None
 
    dicom_info["full_path"] = dicom_info["image_path"].apply(resolve_image_path)
    resolved_info = dicom_info["full_path"].notna().sum()
    print(f"  Images on disk : {resolved_info:,} / {len(dicom_info):,}")
 
    if resolved_info == 0:
        sample = dicom_info["image_path"].iloc[0]
        tried  = os.path.join(
            DATASET_ROOT,
            re.sub(r"^CBIS-DDSM[/\\]", "", str(sample))
        )
        print(f"  Tried path   : {tried}")
        print(f"  DATASET_ROOT : {DATASET_ROOT}")
        print(f"  Contents     : {os.listdir(DATASET_ROOT)}")
        raise RuntimeError(
            "No images resolved. Check DATASET_ROOT points to the "
            "unzipped Kaggle folder containing the jpeg/ subfolder."
        )
 
    # SeriesInstanceUID → full image path
    series_to_path = (
        dicom_info[dicom_info["full_path"].notna()]
        .set_index("SeriesInstanceUID")["full_path"]
        .to_dict()
    )
    print(f"  Series lookup  : {len(series_to_path):,} entries\n")
 
    # ── Step 3: Load descriptor CSVs ─────────────────────────────────────────
    target_names = [
        "mass_case_description_train_set",
        "mass_case_description_test_set",
        "calc_case_description_train_set",
        "calc_case_description_test_set",
    ]
    all_csvs  = glob.glob(os.path.join(DATASET_ROOT, "**", "*.csv"), recursive=True)
    csv_paths = [p for p in all_csvs if any(t in os.path.basename(p) for t in target_names)]
    if not csv_paths:
        raise FileNotFoundError(f"No descriptor CSVs found inside {DATASET_ROOT}.")
 
    dfs = [pd.read_csv(p) for p in csv_paths]
    df  = pd.concat(dfs, ignore_index=True)
    print(f"Descriptor CSVs : {len(csv_paths)} files, {len(df):,} rows")
 
    path_col = next((c for c in df.columns if "pathology" in c.lower()), None)
 
    # ── Step 4: Try all image path columns, pick the best match ──────────────
    path_cols = [
        c for c in df.columns
        if "file path" in c.lower()
        or ("path" in c.lower() and "image" in c.lower())
    ]
 
    def extract_series_uid(dcm_str: str) -> str:
        parts = re.split(r"[/\\]", str(dcm_str).strip())
        return parts[-2] if len(parts) >= 2 else None
 
    print(f"\nTrying {len(path_cols)} image path columns:")
    best_col, best_df, best_count = None, None, 0
    for col in path_cols:
        tmp = df[[col, path_col]].dropna().copy()
        tmp.columns = ["dcm_path", "pathology"]
        tmp["series_uid"] = tmp["dcm_path"].apply(extract_series_uid)
        tmp["img_path"]   = tmp["series_uid"].map(series_to_path)
        n = tmp["img_path"].notna().sum()
        print(f"  {col:40s} → {n:,} resolved")
        if n > best_count:
            best_col, best_df, best_count = col, tmp, n
 
    print(f'\nUsing: "{best_col}" ({best_count:,} images)')
 
    # ── Step 5: Build final dataframe ────────────────────────────────────────
    result = best_df.dropna(subset=["img_path"]).copy()
    result["label"] = result["pathology"].apply(
        lambda x: 1 if str(x).strip().upper() == "MALIGNANT" else 0
    )
    result = result.reset_index(drop=True)
 
    print(f"\n  Total     : {len(result):,}")
    print(f"  Benign    : {(result.label==0).sum():,}  ({(result.label==0).mean()*100:.1f}%)")
    print(f"  Malignant : {(result.label==1).sum():,}  ({(result.label==1).mean()*100:.1f}%)\n")
 
    if len(result) < 500:
        print(
            "⚠️  WARNING: Fewer than 500 images resolved.\n"
            "   Your zip may be a partial Kaggle download.\n"
            "   Use the Kaggle API for a complete download:\n"
            "     pip install kaggle\n"
            "     kaggle datasets download -d "
            "awsaf49/cbis-ddsm-breast-cancer-image-dataset "
            f"--unzip -p \"{DATASET_ROOT}\"\n"
        )
 
    return result

In [4]:
# =============================================================================
# 3. PREPROCESSING — CLAHE (proposal Section 4.4.2)
# =============================================================================
 
def apply_clahe(img_bgr: np.ndarray) -> np.ndarray:
    """
    Contrast Limited Adaptive Histogram Equalization on L-channel.
    Enhances local contrast for mammogram domain adaptation without
    amplifying noise — addresses imaging equipment variability (Section 2.7.1).
    """
    lab          = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe        = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
 
 
# =============================================================================
# 3b. IMAGE CACHE — preprocess once, load fast every epoch
#     CLAHE + resize saved as float32 .npy files (~1.5 GB for 3,568 images).
#     Augmentation still runs on-the-fly so training variety is preserved.
#     Cache survives kernel restarts — built once, used forever.
# =============================================================================
 
def _cache_path(img_path: str) -> str:
    """Return the .npy cache path for a given image path."""
    key = hashlib.md5(img_path.encode()).hexdigest()
    return os.path.join(CACHE_DIR, f"{key}.npy")
 
 
def build_cache(paths: list) -> None:
    """
    Preprocess all images (CLAHE + resize, no augmentation) and save as .npy.
    Skips files that are already cached — safe to call every run.
    """
    missing = [p for p in paths if not os.path.exists(_cache_path(p))]
    if not missing:
        print(f"  Cache up to date — {len(paths):,} files already cached. ✅")
        return
    print(f"  Caching {len(missing):,} images → {CACHE_DIR}")
    print(f"  (This runs once and is skipped on future runs)")
    for p in tqdm(missing, desc="  Building cache"):
        img = cv2.imread(p)
        if img is None:
            arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        else:
            img = apply_clahe(img)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            arr = img.astype(np.float32) / 255.0
        np.save(_cache_path(p), arr)
    print(f"  Cache complete — {len(paths):,} files. ✅\n")
 
 
def load_and_preprocess(img_path: str, augment: bool = False) -> np.ndarray:
    """
    Load from disk cache (CLAHE + resize already done), then apply optional
    OpenCV augmentation → normalised float32 [0,1].
 
    Falls back to full on-the-fly preprocessing on cache miss.
 
    Augmentation (proposal Section 4.4.2):
      - Horizontal flip     (p=0.5)
      - Rotation            ±15°
      - Brightness          ±20%
      - Zoom                0.9–1.1
      - Gaussian noise      σ=0.01
    """
    cp = _cache_path(img_path)
    if os.path.exists(cp):
        img_float = np.load(cp)                           # (224,224,3) float32
        img       = (img_float * 255).astype(np.uint8)    # back to uint8 for cv2
        img       = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)  # augmentation expects BGR
    else:
        # Cache miss — full preprocessing
        img = cv2.imread(img_path)
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        img = apply_clahe(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
 
    if augment:
        # Horizontal flip
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
 
        # Rotation ±15°
        angle = random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((IMG_SIZE // 2, IMG_SIZE // 2), angle, 1)
        img   = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE))
 
        # Brightness ±20%
        hsv          = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.8, 1.2), 0, 255)
        img          = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
 
        # Zoom 0.9–1.1
        zoom   = random.uniform(0.9, 1.1)
        new_sz = int(IMG_SIZE * zoom)
        img    = cv2.resize(img, (new_sz, new_sz))
        if zoom > 1.0:
            s   = (new_sz - IMG_SIZE) // 2
            img = img[s:s + IMG_SIZE, s:s + IMG_SIZE]
        else:
            p   = (IMG_SIZE - new_sz) // 2
            img = cv2.copyMakeBorder(img, p, p, p, p, cv2.BORDER_REFLECT)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
 
        # Gaussian noise σ=0.01
        noise = np.random.normal(0, 0.01 * 255, img.shape).astype(np.float32)
        img   = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
 
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0
 

In [5]:
# =============================================================================
# 4. tf.data PIPELINE
# =============================================================================
 
def make_tf_dataset(
    paths: list,
    labels: list,
    augment: bool = False,
    shuffle: bool = False,
) -> tf.data.Dataset:
 
    def _load(path, label):
        img = tf.numpy_function(
            func=lambda p: load_and_preprocess(p.decode(), augment=augment),
            inp=[path],
            Tout=tf.float32,
        )
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img, label
 
    ds = tf.data.Dataset.from_tensor_slices(
        (paths, np.array(labels, dtype=np.float32))
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
 
    # 8 parallel workers — avoids CPU contention on 24-core machines
    ds = ds.map(_load, num_parallel_calls=8)
 
    # Cache val/test in RAM after first epoch — they never change
    if not augment:
        ds = ds.cache()
 
    return (
        ds.batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE)
    )

In [6]:
# =============================================================================
# 5. MODEL — MobileNetV3-Large (proposal Section 4.4.1)
#    AUPRC added alongside ROC-AUC: more informative under class imbalance,
#    directly surfaces false-negative risk on malignant class (Section 2.7.3)
# =============================================================================
 
def build_model(freeze_base: bool = True):
    base = MobileNetV3Large(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        include_preprocessing=False,
    )
    base.trainable = not freeze_base
 
    reg = tf.keras.regularizers.l2(L2_DECAY)
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=not freeze_base)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(DROPOUT_RATE)(x)
    x   = layers.Dense(256, activation="relu", kernel_regularizer=reg)(x)
    x   = layers.Dropout(DROPOUT_RATE)(x)
    # dtype='float32' keeps output numerically stable under mixed_float16 policy
    out = layers.Dense(1, activation="sigmoid")(x)
 
    return Model(inp, out, name="MobileNetV3_Mammogram"), base
 
 
def compile_model(model: Model) -> None:
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE, beta_1=0.9, beta_2=0.999),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),             # ROC-AUC (H1 primary)
            tf.keras.metrics.AUC(curve="PR", name="prc"), # AUPRC (imbalance-aware)
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

In [7]:
# =============================================================================
# 6. CALLBACKS
#    Both ModelCheckpoint and EarlyStopping monitor val_auc (ROC-AUC)
#    consistent with H1 primary target
# =============================================================================
 
def get_callbacks(checkpoint_path: str) -> list:
    return [
        callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_auc", mode="max",
            save_best_only=True, verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_auc", mode="max",
            patience=PATIENCE_STOP,
            restore_best_weights=True, verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=PATIENCE_LR, min_lr=1e-7, verbose=1,
        ),
        callbacks.TensorBoard(
            log_dir=os.path.join(OUTPUT_DIR, "logs"),
            histogram_freq=1,
        ),
    ]

In [8]:
# =============================================================================
# 7. TRAINING — Two-phase transfer learning (proposal Section 4.4.2)
# =============================================================================
 
def train(train_ds, val_ds, cw_dict: dict, skip_phase1: bool = False):
    model, base = build_model(freeze_base=True)
    compile_model(model)
 
    if skip_phase1:
        # ── Phase 1: Skipped — load Phase 1 best weights ─────────────────────
        print("\n─── Phase 1: Skipped — loading saved checkpoint ───")
        print(f"  Loading weights from : {CHECKPOINT_PATH_P1}")
        model.load_weights(CHECKPOINT_PATH_P1)
        h_frozen = None
    else:
        # ── Phase 1: Train with frozen base ───────────────────────────────────
        print("\n─── Phase 1: Frozen base ───")
        print(f"  Max epochs     : {EPOCHS_FROZEN}")
        print(f"  Early stopping : patience={PATIENCE_STOP} on val_auc")
        print(f"  Checkpoint     : {CHECKPOINT_PATH_P1}\n")
        model.summary(line_length=80)
 
        h_frozen = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_FROZEN,
            class_weight=cw_dict,
            callbacks=get_callbacks(CHECKPOINT_PATH_P1),
            verbose=1,
        )
        # Explicitly reload best Phase 1 weights before entering Phase 2
        print(f"\n  Reloading best Phase 1 weights from : {CHECKPOINT_PATH_P1}")
        model.load_weights(CHECKPOINT_PATH_P1)
 
    # ── Phase 2: Fine-tune last 15 layers ────────────────────────────────────
    print("\n─── Phase 2: Fine-tuning last 15 layers ───")
    print(f"  Checkpoint     : {CHECKPOINT_PATH_P2}\n")
    base.trainable = True
    
    # Changed from -30 to -15
    for layer in base.layers[:-15]:
        layer.trainable = False
        
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Trainable base layers : {trainable}")
 
    # Compile fresh with reduced LR — correct behaviour for fine-tuning
    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE * 0.1,  # 10x lower than Phase 1
            beta_1=0.9, beta_2=0.999,
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(curve="PR", name="prc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    print(f"  Learning rate  : {LEARNING_RATE * 0.1} (Phase 1 was {LEARNING_RATE})")
 
    h_finetune = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINETUNE,
        class_weight=cw_dict,
        callbacks=get_callbacks(CHECKPOINT_PATH_P2),
        verbose=1,
    )
 
    # Reload best Phase 2 weights
    print(f"\n  Reloading best Phase 2 weights from : {CHECKPOINT_PATH_P2}")
    model.load_weights(CHECKPOINT_PATH_P2)
 
    return model, base, h_frozen, h_finetune

In [9]:
# =============================================================================
# 8. TRAINING HISTORY PLOT
# =============================================================================
 
def plot_history(h1, h2) -> None:
    metrics = [
        ("accuracy", "Accuracy"),
        ("loss",     "Loss"),
        ("auc",      "ROC-AUC"),
        ("prc",      "AUPRC"),
    ]
 
    phase1_skipped = h1 is None
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
 
    for ax, (key, title) in zip(axes, metrics):
        if phase1_skipped:
            train_vals = h2.history[key]
            val_vals   = h2.history[f"val_{key}"]
            ax.plot(train_vals, label="Train (Phase 2)")
            ax.plot(val_vals,   label="Validation (Phase 2)")
        else:
            train_vals = h1.history[key] + h2.history[key]
            val_vals   = h1.history[f"val_{key}"] + h2.history[f"val_{key}"]
            split      = len(h1.history["loss"]) - 1
            ax.plot(train_vals, label="Train")
            ax.plot(val_vals,   label="Validation")
            ax.axvline(split, color="gray", linestyle="--",
                       alpha=0.7, label="Fine-tune start")
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
 
    title_str = (
        "MobileNetV3 — Training History (Phase 2 only)"
        if phase1_skipped else
        "MobileNetV3 — Training History (Phase 1 + 2)"
    )
    plt.suptitle(title_str, fontsize=14)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "training_history.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved → {save_path}")

In [10]:
# =============================================================================
# 9. EVALUATION (proposal Section 4.4.3)
#    Reports ROC-AUC + AUPRC with bootstrapped 95% CIs
# =============================================================================
 
def evaluate(model: Model, test_ds, y_te: list) -> dict:
    print("\n─── Test Set Evaluation ───")
    y_prob = model.predict(test_ds, verbose=1).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = np.array(y_te)
 
    roc_auc = roc_auc_score(y_true, y_prob)
    auprc   = average_precision_score(y_true, y_prob)
    report  = classification_report(
        y_true, y_pred, target_names=["Benign", "Malignant"], digits=4
    )
    print(f"\n  ROC-AUC : {roc_auc:.4f}  (H1 target ≥ 0.85)")
    print(f"  AUPRC   : {auprc:.4f}  (imbalance-aware supplement)")
    print(f"\n{report}")
 
    # ── Bootstrapped 95% CI (1,000 iterations) ───────────────────────────────
    print("  Computing bootstrapped 95% CIs (1,000 iterations)...")
    rng = np.random.default_rng(SEED)
    boot_roc, boot_prc = [], []
    for _ in range(1000):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_roc.append(roc_auc_score(y_true[idx], y_prob[idx]))
        boot_prc.append(average_precision_score(y_true[idx], y_prob[idx]))
 
    roc_lo, roc_hi = np.percentile(boot_roc, [2.5, 97.5])
    prc_lo, prc_hi = np.percentile(boot_prc, [2.5, 97.5])
    print(f"  ROC-AUC 95% CI : [{roc_lo:.4f}, {roc_hi:.4f}]")
    print(f"  AUPRC   95% CI : [{prc_lo:.4f}, {prc_hi:.4f}]")
 
    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, label=f"MobileNetV3 (AUC={roc_auc:.3f})")
    axes[0].fill_between(fpr, tpr, alpha=0.1)
    axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve")
    axes[0].legend()
 
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[1].plot(rec, prec, label=f"MobileNetV3 (AUPRC={auprc:.3f})")
    axes[1].fill_between(rec, prec, alpha=0.1)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve")
    axes[1].legend()
 
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"]).plot(
        ax=axes[2], cmap="Blues"
    )
    axes[2].set_title("Confusion Matrix")
 
    plt.suptitle("MobileNetV3 — Mammogram Module Evaluation", fontsize=14)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "evaluation.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"\n  Saved → {save_path}")
 
    return {
        "roc_auc": roc_auc, "roc_lo": roc_lo, "roc_hi": roc_hi,
        "auprc":   auprc,   "prc_lo": prc_lo, "prc_hi": prc_hi,
    }

In [11]:
# =============================================================================
# 10. GRAD-CAM (proposal Section 4.4.3)
#     MobileNetV3 uses DepthwiseConv2D internally — searches both Conv2D
#     and DepthwiseConv2D across the model and any nested submodels.
# =============================================================================
 
def make_gradcam(img_array: np.ndarray, model: Model) -> np.ndarray:
    """
    Compute Grad-CAM heatmap for a single image.
    Fixed: Uses manual eager execution to bypass Keras nested-graph disconnection errors.
    """
    inp = tf.cast(img_array[np.newaxis], tf.float32)

    # 1. Isolate the base model and the head layers
    base_model = None
    head_layers = []
    found_base = False

    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            found_base = True
        elif found_base:
            head_layers.append(layer)

    if base_model is None:
        raise ValueError("No nested Keras submodel found — cannot build Grad-CAM.")

    # 2. Manual forward pass inside the tape
    with tf.GradientTape() as tape:
        # Get spatial features from the base model
        conv_out = base_model(inp, training=False)
        tape.watch(conv_out)

        # Pass through the head layers (GlobalAvgPool, Dropout, Dense)
        x = conv_out
        for layer in head_layers:
            # Dropout behaves differently in training vs inference; strictly enforce False
            if isinstance(layer, tf.keras.layers.Dropout):
                x = layer(x, training=False)
            else:
                x = layer(x)

        preds = x
        class_score = tf.cast(preds[:, 0], tf.float32)

    # 3. Compute gradients
    grads = tape.gradient(class_score, conv_out)
    if grads is None:
        raise ValueError("Gradients are None — tape did not capture conv_out.")
        
    # 4. Cast and process heatmap
    grads    = tf.cast(grads, tf.float32)
    conv_out = tf.cast(conv_out, tf.float32)
    
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap  = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap  = tf.maximum(heatmap, 0)
    
    # Add epsilon to prevent division by zero
    heatmap  = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    
    return heatmap.numpy()
 
 
def overlay_gradcam(img: np.ndarray, heatmap: np.ndarray,
                    alpha: float = 0.4) -> np.ndarray:
    h = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    h = cv2.applyColorMap(np.uint8(255 * h), cv2.COLORMAP_JET)
    h = cv2.cvtColor(h, cv2.COLOR_BGR2RGB)
    return np.uint8(img * 255 * (1 - alpha) + h * alpha)
 
 
def save_gradcam_grid(model: Model, X_te: list, y_te: list) -> None:
    benign_idx    = [i for i, l in enumerate(y_te) if l == 0][:2]
    malignant_idx = [i for i, l in enumerate(y_te) if l == 1][:2]
    sample_idx    = benign_idx + malignant_idx
 
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col, idx in enumerate(sample_idx):
        img   = load_and_preprocess(X_te[idx], augment=False)
        prob  = float(model.predict(img[np.newaxis], verbose=0)[0, 0])
        label = "Malignant" if y_te[idx] == 1 else "Benign"
        tick  = "✓" if int(prob >= 0.5) == y_te[idx] else "✗"
        try:
            hm = make_gradcam(img, model)
            ov = overlay_gradcam(img, hm)
            axes[0, col].imshow(img)
            axes[0, col].set_title(f"Original\n({label})", fontsize=9)
            axes[0, col].axis("off")
            axes[1, col].imshow(ov)
            axes[1, col].set_title(
                f"Grad-CAM  {tick}\nP(malignant)={prob:.2f}", fontsize=9
            )
            axes[1, col].axis("off")
        except Exception as e:
            print(f"  Grad-CAM failed for sample {idx}: {e}")
 
    plt.suptitle(
        "Grad-CAM — Mammogram Module  (col 1-2: benign, col 3-4: malignant)",
        fontsize=12,
    )
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, "gradcam.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved → {save_path}")

In [12]:
# =============================================================================
# 11. INT8 TFLITE QUANTIZATION (proposal Section 4.5.1)
# =============================================================================
 
def quantize_to_tflite(model: Model, cal_paths: list,
                        cal_labels: list) -> tuple:
    print("\n─── INT8 TFLite Quantization ───")
    paths  = cal_paths[:TFLITE_CALIBRATION_SAMPLES]
    labels = cal_labels[:TFLITE_CALIBRATION_SAMPLES]
 
    cal_imgs = np.array(
        [load_and_preprocess(p, augment=False)
         for p in tqdm(paths, desc="  Calibrating")],
        dtype=np.float32,
    )
 
    def representative_dataset():
        for img in cal_imgs:
            yield [img[np.newaxis]]
 
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations          = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type   = tf.uint8
    converter.inference_output_type  = tf.uint8
 
    try:
        tflite_model = converter.convert()
        quant_mode   = "INT8"
        print("  ✓ INT8 quantization successful")
    except Exception as e:
        print(f"  INT8 failed ({e}) — falling back to float16")
        conv2 = tf.lite.TFLiteConverter.from_keras_model(model)
        conv2.optimizations               = [tf.lite.Optimize.DEFAULT]
        conv2.target_spec.supported_types = [tf.float16]
        tflite_model = conv2.convert()
        quant_mode   = "float16"
 
    with open(TFLITE_PATH, "wb") as f:
        f.write(tflite_model)
 
    size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)
    print(f"\n  Mode   : {quant_mode}")
    print(f"  Size   : {size_mb:.2f} MB  (target ≤ 15 MB)")
    print(f"  Saved  : {TFLITE_PATH}")
    print(f"  {'✓ Size target MET' if size_mb <= 15 else '⚠ Exceeds 15 MB'}")
 
    # ── Latency benchmark ─────────────────────────────────────────────────────
    # NOTE: This measures latency on the current machine's CPU.
    # H1 target of ≤100ms is for Android ARM — re-test on device for true result.
    print("\n─── Latency Benchmark (CPU inference, 50 runs) ───")
    print("  Note: H1 target ≤100ms is for Android ARM, not this CPU.")
    # Disable XNNPACK — not all MobileNetV3 INT8 ops are supported
    interpreter = tf.lite.Interpreter(
        model_path=TFLITE_PATH,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    interpreter.allocate_tensors()
    in_det   = interpreter.get_input_details()[0]
    out_det  = interpreter.get_output_details()[0]
    in_dtype = in_det["dtype"]
 
    test_img = load_and_preprocess(paths[0], augment=False)
    test_inp = (
        (test_img * 255).astype(np.uint8)[np.newaxis]
        if in_dtype == np.uint8 else test_img[np.newaxis]
    )
 
    run_times = []
    for _ in range(50):
        t0 = time.perf_counter()
        interpreter.set_tensor(in_det["index"], test_inp)
        interpreter.invoke()
        _ = interpreter.get_tensor(out_det["index"])
        run_times.append((time.perf_counter() - t0) * 1000)
 
    mean_ms = np.mean(run_times[5:])
    p95_ms  = np.percentile(run_times[5:], 95)
    print(f"  Mean : {mean_ms:.1f} ms")
    print(f"  P95  : {p95_ms:.1f} ms")
    print(f"  {'✓ Under 100ms on CPU too' if mean_ms <= 100 else '⚠ Exceeds 100ms on CPU — re-test on Android'}")
 
    # ── Accuracy drop check ───────────────────────────────────────────────────
    print("\n─── Accuracy Drop Check (≤ 2% threshold) ───")
    cal_ds    = make_tf_dataset(paths, labels)
    orig_prob = model.predict(cal_ds, verbose=0).ravel()
    orig_pred = (orig_prob >= 0.5).astype(int)
 
    interpreter2 = tf.lite.Interpreter(
        model_path=TFLITE_PATH,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    interpreter2.allocate_tensors()
    in_det2   = interpreter2.get_input_details()[0]
    out_det2  = interpreter2.get_output_details()[0]
    in_dtype2 = in_det2["dtype"]
 
    quant_pred = []
    for img in cal_imgs:
        inp = (
            (img * 255).astype(np.uint8)[np.newaxis]
            if in_dtype2 == np.uint8 else img[np.newaxis]
        )
        interpreter2.set_tensor(in_det2["index"], inp)
        interpreter2.invoke()
        out  = interpreter2.get_tensor(out_det2["index"])
        prob = out[0, 0] / 255.0 if in_dtype2 == np.uint8 else float(out[0, 0])
        quant_pred.append(int(prob >= 0.5))
 
    orig_acc  = np.mean(orig_pred == np.array(labels))
    quant_acc = np.mean(np.array(quant_pred) == np.array(labels))
    drop      = (orig_acc - quant_acc) * 100
    print(f"  Original accuracy  : {orig_acc:.4f}")
    print(f"  Quantized accuracy : {quant_acc:.4f}")
    print(f"  Drop               : {drop:.2f}%  (threshold ≤ 2.0%)")
    print(f"  {'✓ Within tolerance' if drop <= 2 else '⚠ Exceeds — consider QAT'}")
 
    return size_mb, mean_ms

In [13]:
# =============================================================================
# 12. MAIN PIPELINE
# =============================================================================
 
def main():
    print("─── Step 1 / 5 : Loading dataset ───")
    df     = load_dataset()
    paths  = df["img_path"].tolist()
    labels = df["label"].tolist()
 
    # ── Stratified 70 / 15 / 15 split ────────────────────────────────────────
    print("─── Step 2 / 5 : Splitting data ───")
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        paths, labels,
        test_size=1 - TRAIN_RATIO, stratify=labels, random_state=SEED,
    )
    val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp,
        test_size=1 - val_frac, stratify=y_tmp, random_state=SEED,
    )
    print(f"  Train : {len(X_tr):4d}  ({sum(y_tr)} malignant, {len(y_tr)-sum(y_tr)} benign)")
    print(f"  Val   : {len(X_val):4d}  ({sum(y_val)} malignant, {len(y_val)-sum(y_val)} benign)")
    print(f"  Test  : {len(X_te):4d}  ({sum(y_te)} malignant, {len(y_te)-sum(y_te)} benign)\n")
 
    cw      = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
    cw_dict = {i: float(w) for i, w in enumerate(cw)}
    print(f"  Class weights : {cw_dict}\n")
 
    # ── Build image cache ─────────────────────────────────────────────────────
    # CLAHE + resize done once, saved to disk. Skipped automatically if done.
    print("─── Step 2b: Building image cache ───")
    build_cache(paths)
 
    train_ds = make_tf_dataset(X_tr,  y_tr,  augment=True,  shuffle=True)
    val_ds   = make_tf_dataset(X_val, y_val, augment=False, shuffle=False)
    test_ds  = make_tf_dataset(X_te,  y_te,  augment=False, shuffle=False)
 
    # ── Train ─────────────────────────────────────────────────────────────────
    print("─── Step 3 / 5 : Training ───")
    model, _, h_frozen, h_finetune = train(train_ds, val_ds, cw_dict,
                                           skip_phase1=False)
    plot_history(h_frozen, h_finetune)
 
    # ── Evaluate ──────────────────────────────────────────────────────────────
    print("\n─── Step 4 / 5 : Evaluation ───")
    results = evaluate(model, test_ds, y_te)
 
    print("\n─── Grad-CAM visualisations ───")
    save_gradcam_grid(model, X_te, y_te)
 
    # ── Quantize ──────────────────────────────────────────────────────────────
    print("\n─── Step 5 / 5 : TFLite INT8 Quantization ───")
    size_mb, mean_ms = quantize_to_tflite(model, X_tr, y_tr)
 
    # ── Final summary ─────────────────────────────────────────────────────────
    h1_auc     = results["roc_auc"] >= 0.85
    h1_size    = size_mb  <= 15
    h1_latency = mean_ms  <= 100
    h1_all     = h1_auc and h1_size and h1_latency
 
    print(f"\n{'='*62}")
    print("  PIPELINE COMPLETE")
    print(f"{'='*62}")
    print(f"  ROC-AUC  : {results['roc_auc']:.4f}  "
          f"95% CI [{results['roc_lo']:.4f}, {results['roc_hi']:.4f}]")
    print(f"  AUPRC    : {results['auprc']:.4f}  "
          f"95% CI [{results['prc_lo']:.4f}, {results['prc_hi']:.4f}]")
    print(f"  Size     : {size_mb:.2f} MB")
    print(f"  Latency  : {mean_ms:.1f} ms  (re-test on Android for H1 check)")
    print()
    print("  Hypothesis H1 targets:")
    print(f"    ROC-AUC ≥ 0.85  → {'✓ SUPPORTED' if h1_auc     else '✗ NOT MET'}")
    print(f"    Size    ≤ 15 MB → {'✓ SUPPORTED' if h1_size    else '✗ NOT MET'}")
    print(f"    Latency ≤ 100ms → {'✓ SUPPORTED' if h1_latency else '✗ NOT MET (CPU — re-test on Android)'}")
    print(f"\n  Overall H1 verdict : {'✓ SUPPORTED' if h1_all else '✗ NOT FULLY MET'}")
    print()
    print("  Output files:")
    print(f"    {CHECKPOINT_PATH_P1}")
    print(f"    {CHECKPOINT_PATH_P2}")
    print(f"    {TFLITE_PATH}")
    print(f"    {PLOT_DIR}{os.sep}")
    print(f"{'='*62}\n")
 
 
if __name__ == "__main__":
    main()

─── Step 1 / 5 : Loading dataset ───
dicom_info.csv : 10,237 rows
  Images on disk : 10,237 / 10,237
  Series lookup  : 6,774 entries

Descriptor CSVs : 4 files, 3,568 rows

Trying 3 image path columns:
  image file path                          → 3,568 resolved
  cropped image file path                  → 3,567 resolved
  ROI mask file path                       → 3,567 resolved

Using: "image file path" (3,568 images)

  Total     : 3,568
  Benign    : 2,111  (59.2%)
  Malignant : 1,457  (40.8%)

─── Step 2 / 5 : Splitting data ───
  Train : 2497  (1020 malignant, 1477 benign)
  Val   :  535  (218 malignant, 317 benign)
  Test  :  536  (219 malignant, 317 benign)

  Class weights : {0: 0.8452945159106297, 1: 1.2240196078431373}

─── Step 2b: Building image cache ───
  Cache up to date — 3,568 files already cached. ✅


I0000 00:00:1776140915.820553   20547 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9694 MB memory:  -> device: 0, name: NVIDIA RTX A3000 12GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


─── Step 3 / 5 : Training ───

─── Phase 1: Frozen base ───
  Max epochs     : 100
  Early stopping : patience=15 on val_auc
  Checkpoint     : /home/mmando/mammogram_output/best_mobilenetv3_phase1.keras



Model: "MobileNetV3_Mammogram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)        │ (None, 224, 224, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)     │ (None, 7, 7, 960)        │     2,996,352 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ global_average_pooling2d          │ (None, 960)              │             0 │
│ (GlobalAveragePooling2D)          │                          │               │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 960)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 256)              │       246,016 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_1 (Dropout)               │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_1 (Dense)                   │ (None, 1)                │           257 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

...

Saved artifact at '/tmp/tmp189wp4m7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_202')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  128606500437648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291525712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291527632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291526864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291526288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291525904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291527824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291528592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291528208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128606291526672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1286062915

W0000 00:00:1776142157.683307   20547 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776142157.683355   20547 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776142157.683668   20547 reader.cc:83] Reading SavedModel from: /tmp/tmp189wp4m7
I0000 00:00:1776142157.692341   20547 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776142157.692355   20547 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp189wp4m7
I0000 00:00:1776142157.782418   20547 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1776142157.797917   20547 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776142158.295769   20547 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp189wp4m7
I0000 00:00:1776142158.420084   20547 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 736428 microseconds.
fully_quantize: 0, inference_type: 6

  ✓ INT8 quantization successful

  Mode   : INT8
  Size   : 3.60 MB  (target ≤ 15 MB)
  Saved  : /home/mmando/mammogram_output/mammogram_mobilenetv3_int8.tflite
  ✓ Size target MET

─── Latency Benchmark (CPU inference, 50 runs) ───
  Note: H1 target ≤100ms is for Android ARM, not this CPU.
  Mean : 24.5 ms
  P95  : 27.4 ms
  ✓ Under 100ms on CPU too

─── Accuracy Drop Check (≤ 2% threshold) ───


E0000 00:00:1776142258.200285   20717 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1776142258.705995   20717 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  Original accuracy  : 0.6580
  Quantized accuracy : 0.4770
  Drop               : 18.10%  (threshold ≤ 2.0%)
  ⚠ Exceeds — consider QAT

  PIPELINE COMPLETE
  ROC-AUC  : 0.7060  95% CI [0.6614, 0.7483]
  AUPRC    : 0.5540  95% CI [0.4940, 0.6205]
  Size     : 3.60 MB
  Latency  : 24.5 ms  (re-test on Android for H1 check)

  Hypothesis H1 targets:
    ROC-AUC ≥ 0.85  → ✗ NOT MET
    Size    ≤ 15 MB → ✓ SUPPORTED
    Latency ≤ 100ms → ✓ SUPPORTED

  Overall H1 verdict : ✗ NOT FULLY MET

  Output files:
    /home/mmando/mammogram_output/best_mobilenetv3_phase1.keras
    /home/mmando/mammogram_output/best_mobilenetv3_phase2.keras
    /home/mmando/mammogram_output/mammogram_mobilenetv3_int8.tflite
    /home/mmando/mammogram_output/plots/

